# Trading Intelligent — CH01 — NB01
## Market Data, OHLCV & Microstructure

**Auteur académique : Tawfik Masrour**  
**Édition : CH01 v2.1**

> **Question directrice :** Que contient réellement une donnée de marché, et qu'a-t-on déjà perdu lorsque l'on observe une bougie ?

### Ce que vous allez apprendre
- distinguer transaction, cotation, best bid, best ask, mid-price et last price ;
- lire une bougie OHLC et expliquer ce que l'agrégation ne conserve pas ;
- visualiser la compression du flux fin vers des barres plus larges ;
- interpréter un gap et un volume élevé sans règle simpliste ;
- distinguer observation, fenêtre historique, horizon et rebalancement.

### Prérequis
Aisance élémentaire avec Python et `pandas`. Aucune connaissance préalable de microstructure n'est supposée : les notions financières utiles sont introduites au moment où elles deviennent nécessaires.

### Comment utiliser ce notebook
Lisez la **question** avant d'exécuter la cellule Python. Formulez d'abord une réponse ou une prédiction, puis comparez-la à la sortie et à son interprétation. Les mini-défis servent à contrôler la compréhension, pas seulement la capacité à faire exécuter du code.

Ce notebook est un **laboratoire interactif de textbook** : question → intuition → expérience → observation → interprétation.

### 0. Préparer un environnement reproductible

Le notebook utilise **TI_CH01_MARKET_V1**. Il cherche d'abord le CSV distribué avec le chapitre ; s'il ne le trouve pas, il reconstruit exactement le même jeu à partir d'une graine fixe. Aucun accès Internet n'est requis.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Rectangle

DATASET_ID = "TI_CH01_MARKET_V1"
DATASET_FILENAME = "TI_CH01_Market_Data_v1.csv"
SEED = 20260929

# Palette commune au PDF et aux notebooks.
ACADEMIC_BLUE = "#173B57"
PETROL = "#2C7A7B"
WARM_GOLD = "#C79A3B"
MUTED_RED = "#A94B4B"
NEUTRAL_GRAY = "#66717D"
PALE_BLUE = "#EAF1F5"

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "semibold",
})
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)


def generate_ti_ch01_market_v1(seed=SEED):
    """Reproduit exactement le jeu pédagogique synthétique TI_CH01_MARKET_V1."""
    rng = np.random.default_rng(seed)
    dates = pd.bdate_range("2023-01-02", periods=756)
    n = len(dates)
    vol = np.full(n, 0.010)
    vol[170:260] = 0.022
    vol[260:430] = 0.009
    vol[430:515] = 0.028
    vol[515:] = 0.012
    vol = pd.Series(vol).rolling(7, center=True, min_periods=1).mean().to_numpy()
    mu = np.full(n, 0.00020)
    mu[430:515] = -0.00035
    z = rng.standard_t(df=6, size=n) / np.sqrt(6/(6-2))
    log_ret = mu + vol * z
    close = 100 * np.exp(np.cumsum(log_ret))

    overnight = rng.normal(0, vol * 0.22)
    open_ = np.empty(n)
    open_[0] = 100 * np.exp(overnight[0])
    open_[1:] = close[:-1] * np.exp(overnight[1:])
    range_scale = np.maximum(vol, np.abs(log_ret)) * (0.8 + 0.6*rng.random(n))
    high = np.maximum(open_, close) * (1 + range_scale * (0.45 + 0.35*rng.random(n)))
    low = np.minimum(open_, close) * (1 - range_scale * (0.45 + 0.35*rng.random(n)))
    low = np.maximum(low, 0.01)
    volume = np.exp(16.3 + 11*np.abs(log_ret) + rng.normal(0, 0.28, n)).astype("int64")
    spread_bps = 3.0 + 85*vol + 35*np.abs(log_ret) + rng.gamma(1.5, 0.4, n)
    spread = close * spread_bps / 10000
    bid = close - spread/2
    ask = close + spread/2

    return pd.DataFrame({
        "date": dates, "symbol": "SYNTH-MKT",
        "open": open_, "high": high, "low": low, "close": close,
        "adjusted_close": close, "volume": volume, "bid": bid, "ask": ask,
    })


def locate_dataset(filename=DATASET_FILENAME):
    cwd = Path.cwd().resolve()
    roots = [cwd, *cwd.parents[:4]]
    subdirs = [Path("."), Path("datasets"), Path("data"), Path("resources")]
    seen = set()
    for root in roots:
        for sub in subdirs:
            candidate = (root / sub / filename).resolve()
            if candidate in seen:
                continue
            seen.add(candidate)
            if candidate.exists():
                return candidate
    return None

DATA_PATH = locate_dataset()
if DATA_PATH:
    df = pd.read_csv(DATA_PATH, parse_dates=["date"])
    source_msg = f"{DATASET_ID} chargé depuis {DATA_PATH}"
else:
    df = generate_ti_ch01_market_v1()
    source_msg = f"{DATASET_ID} régénéré en mémoire (fallback déterministe)"

df = df.sort_values("date").reset_index(drop=True)
print(source_msg)
print("Dimensions :", df.shape)

### 1. Un marché réel n'est pas un tableau

**Question.** Que représente réellement une ligne du fichier ?

Une ligne quotidienne est déjà une **agrégation**. Le flux réel contient des événements plus fins : transactions, changements de cotation, ordres et annulations. Commençons par regarder le schéma sans calculer d'indicateur.

In [ ]:
display(df.head(6))
print(df.dtypes)
print("Période :", df["date"].min().date(), "→", df["date"].max().date())

**Interprétation.** `open`, `high`, `low`, `close` et `volume` décrivent une période. `bid` et `ask` décrivent ici les meilleures cotations d'une représentation simplifiée. Une colonne n'est pas seulement un nombre : elle possède une convention économique et temporelle.

### 2. Transaction, cotation, bid, ask, mid et last

**Question.** Pourquoi ne peut-on pas parler d'un prix unique au même instant ?

Le **best bid** est le plus haut prix acheteur actuellement disponible ; le **best ask** est le plus bas prix vendeur actuellement disponible. Le **mid-price** est leur milieu. Le **last price** est le prix de la dernière transaction observée.

Pour enseigner la différence sans ambiguïté, utilisons une micro-situation où les quatre valeurs sont réellement distinctes.

In [ ]:
bid = 99.95
ask = 100.05
mid = (bid + ask) / 2
last = 99.98
spread = ask - bid
spread_bps = 10_000 * spread / mid

print(f"Best bid : {bid:.2f}")
print(f"Best ask : {ask:.2f}")
print(f"Mid      : {mid:.2f}")
print(f"Last     : {last:.2f}")
print(f"Spread   : {spread:.2f} soit {spread_bps:.2f} bps")
print("Les quatre valeurs sont distinctes :", len({bid, ask, mid, last}) == 4)

**Interprétation.** `Last = 99.98` décrit une transaction passée. `Bid = 99.95` et `Ask = 100.05` décrivent les meilleures cotations accessibles dans cet exemple. `Mid = 100.00` est un prix construit : il n'est pas nécessairement exécutable immédiatement.

### 3. Coût de transaction et PnL : deux signes à ne pas confondre

**Question.** Que perd-on si le marché ne bouge pas mais que l'on achète puis revend immédiatement ?

Si l'on appelle une quantité **coût**, on l'exprime positivement. Le PnL du même aller-retour est négatif.

In [ ]:
entry = ask
exit_ = bid
transaction_cost = ask - bid
transaction_cost_bps = 10_000 * transaction_cost / mid
pnl = exit_ - entry
pnl_bps = 10_000 * pnl / mid

print(f"Achat à l'ask    : {entry:.2f}")
print(f"Revente au bid   : {exit_:.2f}")
print(f"Coût             : +{transaction_cost:.2f} = +{transaction_cost_bps:.2f} bps")
print(f"PnL aller-retour : {pnl:.2f} = {pnl_bps:.2f} bps")
assert transaction_cost > 0 and pnl < 0

**À retenir.** Le même spread peut apparaître comme un **coût positif** ou comme un **PnL négatif** selon l'objet mesuré. La convention de signe doit être écrite explicitement. Slippage, frais et profondeur du carnet seront traités plus tard.

### 4. Anatomie d'une bougie OHLC

**Question.** Que signifient réellement corps, mèches, Open, High, Low et Close ?

Prenons `O=100`, `H=104`, `L=98`, `C=102`. La bougie est haussière car `C > O` ; son corps relie 100 à 102, et ses mèches atteignent 104 et 98.

In [ ]:
O,H,L,C = 100,104,98,102
fig, ax = plt.subplots(figsize=(5,5))
ax.vlines(1, L, H, lw=2, color=PETROL)
ax.add_patch(Rectangle((0.72,min(O,C)),0.56,abs(C-O),alpha=.82,
                       facecolor=PETROL,edgecolor=PETROL))
for y,label in [(H,'High'),(C,'Close'),(O,'Open'),(L,'Low')]:
    ax.annotate(label,(1,y),xytext=(1.55,y),color=ACADEMIC_BLUE,
                arrowprops=dict(arrowstyle='->',color=NEUTRAL_GRAY))
ax.set_xlim(.2,2.4); ax.set_ylim(96.8,105.2); ax.set_xticks([]); ax.set_ylabel('Prix')
ax.set_title('Bougie haussière : O=100, H=104, L=98, C=102')
ax.grid(axis='y',alpha=.2); plt.show()

**Observation.** La figure localise les quatre prix résumés. Elle ne dit pas dans quel ordre High et Low ont été touchés.

### 5. Deux histoires différentes, exactement le même OHLC

**Question.** Deux périodes ayant le même OHLC ont-elles nécessairement suivi la même trajectoire ?

In [ ]:
path_A = pd.Series([100,98,104,102], index=['O','L','H','C'])
path_B = pd.Series([100,104,98,102], index=['O','H','L','C'])

def ohlc(values):
    a = np.asarray(values,dtype=float)
    return {'Open':a[0],'High':a.max(),'Low':a.min(),'Close':a[-1]}

print('OHLC A :', ohlc(path_A.values))
print('OHLC B :', ohlc(path_B.values))
fig, ax = plt.subplots(figsize=(9,4))
ax.plot(range(4),path_A.values,marker='o',color=PETROL,label='A : O → L → H → C')
ax.plot(range(4),path_B.values,marker='s',color=MUTED_RED,label='B : O → H → L → C')
ax.set_xticks(range(4),['début','étape 1','étape 2','fin'])
ax.set_ylabel('Prix'); ax.set_title('Même OHLC, ordre intrapériode différent')
ax.legend(); ax.grid(alpha=.2); plt.show()

**Conséquence pour le trading.** Un stop-loss et un take-profit situés dans la même barre peuvent donner des résultats différents selon l'ordre intrapériode. Une barre OHLC compresse l'information.

### 6. Du tick à la barre : voir littéralement la compression

**Question.** Que perd-on quand on passe d'un flux fin à des barres de 1, 5 puis 60 minutes ?

Nous créons un flux synthétique intrajournalier uniquement pour isoler le mécanisme d'agrégation.

In [ ]:
rng=np.random.default_rng(123)
ts=pd.date_range('2026-09-30 09:00', periods=6*60*4, freq='15s')
ret=rng.normal(0,0.00035,len(ts))
price=100*np.exp(np.cumsum(ret))
size=rng.integers(1,80,len(ts))
ticks=pd.DataFrame({'timestamp':ts,'price':price,'size':size}).set_index('timestamp')

def make_bars(freq):
    px=ticks['price'].resample(freq).ohlc()
    vol=ticks['size'].resample(freq).sum().rename('volume')
    return px.join(vol).dropna()

bars_1m=make_bars('1min')
bars_5m=make_bars('5min')
bars_60m=make_bars('60min')
print('Ticks        :',len(ticks))
print('Barres 1 min :',len(bars_1m))
print('Barres 5 min :',len(bars_5m))
print('Barres 60 min:',len(bars_60m))
display(bars_5m.head())

In [ ]:
fig, axes = plt.subplots(4,1,figsize=(11,9),sharex=True)
series = [
    ('Flux fin — ticks 15 s', ticks.index, ticks['price'], ACADEMIC_BLUE, len(ticks)),
    ('Barres 1 min — Close', bars_1m.index, bars_1m['close'], PETROL, len(bars_1m)),
    ('Barres 5 min — Close', bars_5m.index, bars_5m['close'], WARM_GOLD, len(bars_5m)),
    ('Barres 60 min — Close', bars_60m.index, bars_60m['close'], MUTED_RED, len(bars_60m)),
]
for ax,(title,idx,values,col,nobs) in zip(axes,series):
    ax.plot(idx,values,lw=1.0,marker='o' if nobs < 100 else None,ms=2.8,color=col)
    ax.set_ylabel('Prix')
    ax.set_title(f'{title} — {nobs} observations',loc='left',fontsize=10)
    ax.grid(alpha=.18)
axes[-1].set_xlabel('Même période de six heures')
fig.suptitle("Même période, résolutions différentes : l'agrégation compresse l'information",y=.995)
fig.tight_layout(); plt.show()

**Interprétation.** La période est exactement la même dans les quatre panneaux. En élargissant les barres, de nombreux événements sont remplacés par quelques résumés OHLC : l'ordre intrapériode disparaît progressivement. Cela peut modifier la simulation d'un stop-loss, d'un take-profit ou d'une exécution. Une fréquence plus fine n'est toutefois pas automatiquement « meilleure » : elle contient aussi davantage de bruit de microstructure.

### 7. Close-to-open et gaps : deux notions à distinguer

**Question.** Où apparaissent les plus grands écarts entre la clôture précédente et l'ouverture suivante ?

In [ ]:
df['close_to_open_pct']=100*(df['open']/df['close'].shift(1)-1)
cols=['date','open','close','close_to_open_pct']
display(df.loc[df['close_to_open_pct'].abs().nlargest(8).index,cols].sort_values('date'))
print('Écart close-to-open médian absolu (%) :',round(df['close_to_open_pct'].abs().median(),4))

**Interprétation.** Le calcul mesure un rendement **close-to-open**. On parle couramment de **gap** lorsqu'une discontinuité significative apparaît entre les zones de prix de deux périodes successives ; toute différence microscopique n'est donc pas nécessairement un gap visuel. Dans ce jeu synthétique, les écarts proviennent de la composante overnight simulée.

### 8. Prix et volume : regarder ensemble sans surinterpréter

**Question.** Un volume élevé « valide-t-il » automatiquement une tendance ?

Non. Le volume mesure d'abord une activité selon une unité et une convention. Le panneau supérieur décrira les niveaux de prix ; le panneau inférieur, la quantité échangée.

In [ ]:
s=df.iloc[440:470].copy(); x=mdates.date2num(s['date'])
fig=plt.figure(figsize=(10,6))
ax=fig.add_axes([.10,.36,.86,.56]); av=fig.add_axes([.10,.12,.86,.18],sharex=ax)
for xi,o,h,l,c in zip(x,s.open,s.high,s.low,s.close):
    col=PETROL if c>=o else MUTED_RED
    ax.vlines(xi,l,h,lw=.9,color=col)
    ax.add_patch(Rectangle((xi-.28,min(o,c)),.56,max(abs(c-o),.04),
                           alpha=.78,facecolor=col,edgecolor=col))
av.bar(x,s.volume,width=.65,alpha=.58,color=ACADEMIC_BLUE)
ax.set_ylabel('Prix'); av.set_ylabel('Volume'); av.set_xlabel('Date')
ax.grid(axis='y',alpha=.15); av.grid(axis='y',alpha=.15)
ax.set_title('Bougies et activité échangée : deux informations distinctes')
av.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
plt.setp(ax.get_xticklabels(),visible=False); plt.show()

**Piège.** Un volume inhabituel peut accompagner une hausse, une baisse, un retournement ou une période de désaccord intense. Un volume élevé signifie d'abord qu'une activité d'échange importante a eu lieu **relativement à une référence choisie** ; son interprétation directionnelle exige des informations supplémentaires.

### 9. Les quatre horloges

**Question.** Peut-on observer chaque minute, regarder 20 jours de passé, prédire à 3 heures et agir toutes les 30 minutes ?

Oui. Ces quatre temporalités répondent à quatre questions différentes.

In [ ]:
clocks=pd.DataFrame({
    "Objet":["Fréquence d'observation","Fenêtre historique","Horizon de prédiction","Fréquence de rebalancement"],
    "Exemple":["1 minute","20 jours","3 heures","30 minutes"],
    "Question":["Quand mesure-t-on ?","Quelle profondeur de passé ?","Jusqu'où vise-t-on ?","Quand peut-on agir ?"]
})
display(clocks)

**À retenir.** Une fenêtre historique de 20 jours n'implique ni une prédiction à 20 jours, ni une position conservée 20 jours. Capteurs, rétroviseur, phares et volant forment une analogie utile ; les définitions temporelles restent la référence.

### 10. Mini-défis

1. Construisez une troisième trajectoire ayant `O=100, H=104, L=98, C=102`.
2. Calculez, pour dix lignes du dataset, le **coût positif** d'un aller-retour en bps et le **PnL négatif** correspondant.
3. Comparez le nombre de barres obtenu avec 30 secondes, 2 minutes et 15 minutes.
4. Repérez un écart close-to-open important et expliquez pourquoi il n'est pas automatiquement une erreur de donnée.

In [ ]:
# Votre travail commence ici.
# Q1

# Q2

# Q3

# Q4

### 11. Synthèse

- Le marché n'est pas un CSV : le fichier est déjà une représentation.
- Best bid, best ask, mid et last ne sont pas interchangeables.
- Coût et PnL exigent une convention de signe explicite.
- Une bougie décrit corps et mèches, mais pas l'ordre intrapériode.
- L'agrégation détruit de l'information ; la fréquence est donc un choix scientifique.
- Close-to-open, gap et volume exigent une interprétation précise.
- Observation, fenêtre historique, horizon et rebalancement sont quatre horloges différentes.

### Checklist de maîtrise

Avant de considérer NB01 acquis, vérifiez que vous pouvez répondre **oui** sans regarder les cellules précédentes :

- [ ] Je peux expliquer best bid, best ask, mid-price et last price sans les confondre.
- [ ] Je peux expliquer pourquoi un coût d'aller-retour est positif alors que le PnL immédiat correspondant est négatif.
- [ ] Je peux lire Open, High, Low et Close sur une bougie et expliquer corps et mèches.
- [ ] Je peux construire deux trajectoires différentes ayant le même OHLC et dire ce que l'agrégation a détruit.
- [ ] Je peux expliquer ce qu'un volume élevé indique — et ce qu'il n'indique pas à lui seul.
- [ ] Je peux distinguer fréquence d'observation, fenêtre historique, horizon et rebalancement.